# Week 2 Day 2 — Gradio (Anthropic edition)

Build UIs with Gradio. The Gradio parts are identical to the original; only the model calls change to Claude. Where the original offered a GPT/Claude toggle, this version offers **Claude / Ollama (local)** so both options are available to you with just your Anthropic key.

In [1]:
import os
from dotenv import load_dotenv
import anthropic
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
print("Anthropic key set" if anthropic_api_key else "Anthropic key NOT set")

client = anthropic.Anthropic()                                   # native Claude
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")  # free local

C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Anthropic key set


In [2]:
system_message = "You are a helpful assistant"

def message_claude(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=1000,
        system=system_message,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

In [3]:
message_claude("What is the date today?")

"I don't have access to real-time information, so I can't tell you today's exact date. You can check the date on your device's clock, calendar, or by doing a quick search online."

## A first Gradio interface

In [4]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Shout has been called with input sam


In [6]:
# Wire the model into a Gradio UI
message_input = gr.Textbox(label="Your message:", lines=7)
message_output = gr.Markdown(label="Response:")

gr.Interface(fn=message_claude, title="Claude", inputs=[message_input], outputs=[message_output],
             examples=["Explain the Transformer architecture to a layperson"],
             flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Streaming

Anthropic streaming uses `with client.messages.stream(...) as stream:` and iterating `stream.text_stream`. In Gradio we `yield` the accumulating text.

In [7]:
def stream_claude(prompt):
    with client.messages.stream(
        model="claude-sonnet-4-6", max_tokens=1000,
        system=system_message,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        result = ""
        for text in stream.text_stream:
            result += text
            yield result

In [8]:
def stream_ollama(prompt):
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[{"role": "system", "content": system_message}, {"role": "user", "content": prompt}],
        stream=True,
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [9]:
def stream_model(prompt, model):
    if model == "Claude":
        yield from stream_claude(prompt)
    elif model == "Ollama":
        yield from stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")

In [10]:
message_input = gr.Textbox(label="Your message:", lines=7)
model_selector = gr.Dropdown(["Claude", "Ollama"], label="Select model", value="Claude")
message_output = gr.Markdown(label="Response:")

gr.Interface(fn=stream_model, title="LLMs",
             inputs=[message_input, model_selector], outputs=[message_output],
             examples=[["Explain the Transformer architecture to a layperson", "Claude"]],
             flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Brochure generator with a UI

Reusing the Week 1 scraper. Keep `scraper.py` in this folder.

In [11]:
from scraper import fetch_website_contents

In [12]:
brochure_system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

def stream_brochure(company_name, url, model):
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    # swap the system message in for the brochure task
    global system_message
    system_message = brochure_system_message
    if model == "Claude":
        yield from stream_claude(prompt)
    elif model == "Ollama":
        yield from stream_ollama(prompt)

In [13]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including https://")
model_selector = gr.Dropdown(["Claude", "Ollama"], label="Select model", value="Claude")
message_output = gr.Markdown(label="Response:")

gr.Interface(fn=stream_brochure, title="Brochure Generator",
             inputs=[name_input, url_input, model_selector], outputs=[message_output],
             examples=[["Hugging Face", "https://huggingface.co", "Claude"]],
             flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


C:\Users\Sanjit Kulkarni\AppData\Roaming\Python\Python314\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
